# Browsing a sweep

`box.ipynb` and `column.ipynb` browse **one** CrunchTope run. This browses a **sweep** -- many runs
that differ in whatever was varied -- and the difference matters for how it is laid out.

A sweep is run to compare across runs, so the run axis is drawn **whole**: every plot here shows all
the runs at once and the widgets choose *which variable* to look at, not *which run*. Stepping
through runs one at a time would hide the very thing the sweep was run to show. The exception is the
2-D map at the end, which can only show one run.

**The widgets are a convenience, not a dependency.** Every control below does nothing but call a
function from `coeus.sweep_plots`. If the widget stack misbehaves, call those functions directly --
the last cell shows how -- and you lose the sliders, not the analysis.

## Canary

Run this first. If you do not see a slider, the widget stack is not working in this kernel and
nothing below will draw. Skip to the last cell and call the plotting functions directly.

Widgets need `ipywidgets` **and** a kernel that can render it. Use the `JupyterEnv` kernel: the
`topepan` environment has ipywidgets but no `ipykernel`, so it cannot act as one.

In [ ]:
import ipywidgets as widgets

print(f'ipywidgets {widgets.__version__}')
widgets.IntSlider(description='canary')

## Load a sweep

`describe` is worth reading before any plotting. It says how many runs there are, what was varied,
and -- the part that is easy to miss -- whether the runs actually finished. A sweep whose runs time
out still writes a `results.nc` full of plausible numbers.

In [ ]:
import os
import sys
from pathlib import Path


def find_omphalos():
    """Locate the Omphalos checkout so `coeus` can be imported.

    Not a fixed path: topepan is used on more than one machine and Omphalos does not always sit in
    the same place. Preference order is an explicit OMPHALOS_DIR, then a sibling directory of
    wherever this notebook is running, which is the usual layout -- topepan/ and Omphalos/ next to
    each other in a CrunchTope working directory.

    Returns None if Omphalos is already importable, e.g. `pip install -e` into this environment.
    """
    try:
        import coeus.sweep  # noqa: F401
        return None
    except ImportError:
        pass

    if os.environ.get('OMPHALOS_DIR'):
        return Path(os.environ['OMPHALOS_DIR'])

    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / 'Omphalos'
        if (candidate / 'coeus' / 'sweep.py').is_file():
            return candidate

    raise ImportError(
        'Could not find Omphalos, which sweep.ipynb needs for coeus. Either set OMPHALOS_DIR to '
        'the checkout, or install it into this environment with `pip install -e /path/to/Omphalos`.')


found = find_omphalos()

if found is not None:
    sys.path.insert(0, str(found))
    print(f'using Omphalos at {found}')

import matplotlib.pyplot as plt
from coeus.sweep import Sweep, describe
from coeus import sweep_plots as sp

sp.use_style()

# The sweep to read. conditions.nc is expected beside it.
RESULTS = Path('/path/to/your/omphalos/sweep/results.nc')

sweep = Sweep(RESULTS)
describe(sweep)

## Profiles along the column

One line per run, at the chosen output time. The legend names whatever was swept, so the lines say
what distinguishes them rather than merely being numbered.

Where a sweep crosses two parameters, the labels default to whichever separates the most runs. If
that is not the one you care about, pass `parameter=` to `sp.profiles` directly.

In [ ]:
spatial = [g for g in sweep.groups if 'time' in sweep.data(g).dims and 'X' in sweep.data(g).dims]


def show_profiles(group, variable, time, orientation):
    fig, axis = plt.subplots()
    sp.profiles(sweep, group, variable, time=time, axis=axis, vertical=orientation)
    plt.show()


group_pick = widgets.Dropdown(options=sorted(spatial), description='group')
variable_pick = widgets.Dropdown(options=sorted(sweep.data(group_pick.value).data_vars),
                                 description='variable')
# X down the y axis is the depth convention; X across the x axis reads better for a flow
# path. Both are useful for a 1-D column.
orientation_pick = widgets.ToggleButtons(
    options=[('X on x axis', False), ('X on y axis (depth)', True)],
    value=False, description='Orientation')
time_pick = widgets.IntSlider(min=0, max=sweep.data(group_pick.value).sizes['time'] - 1,
                              value=sweep.data(group_pick.value).sizes['time'] - 1,
                              description='time index')


def on_group(change):
    data = sweep.data(change['new'])
    variable_pick.options = sorted(data.data_vars)
    time_pick.max = data.sizes['time'] - 1


group_pick.observe(on_group, names='value')
widgets.interact(show_profiles, group=group_pick, variable=variable_pick, time=time_pick,
                 orientation=orientation_pick);

## Time series at an observation point

The `timeseries_*` groups are written every timestep rather than at the snapshot times, so they show
arrival and breakthrough that the profiles above cannot resolve. Runs that stopped early simply end
early rather than running a flat line to the edge of the axis.

In [ ]:
series_groups = [g for g in sweep.groups if g.startswith('timeseries_')]

if series_groups:
    def show_series(group, variable):
        fig, axis = plt.subplots()
        sp.time_series(sweep, group, variable, axis=axis)
        plt.show()

    series_pick = widgets.Dropdown(options=sorted(series_groups), description='group')
    series_var = widgets.Dropdown(options=sorted(sweep.data(series_pick.value).data_vars),
                                  description='variable')

    series_pick.observe(
        lambda change: setattr(series_var, 'options',
                               sorted(sweep.data(change['new']).data_vars)),
        names='value')

    widgets.interact(show_series, group=series_pick, variable=series_var);
else:
    print('this sweep has no timeseries_* groups')

## One run at a time: the 2-D map

The only view that cannot show the sweep axis whole, so here the run *is* the control. On a 1-D
column this is not worth looking at, and the cell says so rather than drawing a stripe.

In [ ]:
two_d = [g for g in spatial if sweep.data(g).sizes.get('Y', 1) > 1]

if two_d:
    def show_field(group, variable, run, time):
        fig, axis = plt.subplots()
        sp.field(sweep, group, variable, run=run, time=time, axis=axis)
        plt.show()

    field_group = widgets.Dropdown(options=sorted(two_d), description='group')
    field_var = widgets.Dropdown(options=sorted(sweep.data(field_group.value).data_vars),
                                 description='variable')
    run_pick = widgets.IntSlider(min=min(sweep.runs), max=max(sweep.runs), description='run')

    field_group.observe(
        lambda change: setattr(field_var, 'options', sorted(sweep.data(change['new']).data_vars)),
        names='value')

    widgets.interact(show_field, group=field_group, variable=field_var, run=run_pick,
                     time=widgets.IntSlider(min=0, max=sweep.data(field_group.value).sizes['time'] - 1,
                                            value=sweep.data(field_group.value).sizes['time'] - 1,
                                            description='time index'));
else:
    print(f'{RESULTS.name} is 1-D (Y=1), so there is no 2-D field to map')

## Without the widgets

Everything above is one call into `coeus.sweep_plots`. If the widget stack is broken, or you are
working from a script rather than a notebook, call them directly -- and this is also the starting
point for a figure that is going into something.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)

sp.profiles(sweep, 'totcon', 'SO4--', axis=axes[0])
sp.time_series(sweep, 'timeseries_fiftycm', 'SO4--', axis=axes[1], legend=False)
sp.label_panels(axes)

fig.savefig('sweep_overview.png', dpi=200)